# 02 — Sklearn GBM Tuning (AQP × ML4T Ch12)

Mirrors `02_sklearn_gbm_tuning.ipynb`. We grid-search the hyperparameter cube around our `sklearn_gbm_tuning.yaml` baseline and pick the best config by validation IC.


In [ ]:
from pathlib import Path
from itertools import product
import yaml
import pandas as pd
from aqp.core.registry import build_from_config
from aqp.ml.factor_eval import information_coefficient

cfg = yaml.safe_load((Path('../../../configs/ml/ml4t/sklearn_gbm_tuning.yaml')).read_text())
dataset = build_from_config(cfg['dataset'])

GRID = {
    'num_leaves': [31, 63, 127],
    'learning_rate': [0.01, 0.03, 0.05],
    'min_child_samples': [20, 50, 100],
}

rows = []
for nl, lr, mcs in product(*GRID.values()):
    mc = {**cfg['model']}
    mc['kwargs'] = {**mc.get('kwargs', {}), 'num_leaves': nl, 'learning_rate': lr, 'min_child_samples': mcs}
    model = build_from_config(mc)
    model.fit(dataset)
    pred = model.predict(dataset, segment='valid')
    valid = dataset.prepare('valid')[['label']]
    ic = information_coefficient(pred.rename('pred'), valid['label']).mean()
    rows.append({'num_leaves': nl, 'learning_rate': lr, 'min_child_samples': mcs, 'ic': float(ic)})

df = pd.DataFrame(rows).sort_values('ic', ascending=False)
df.head(10)